In [1]:
import random
import time
import re
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "cross-encoder/stsb-distilroberta-base"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"

if torch.backends.mps.is_available():
    device = "mps"
    pipeline_device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
    pipeline_device = 0
else:
    device = "cpu"
    pipeline_device = -1

batch_size = 64 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "pipeline_device": pipeline_device,
    "batch_size": batch_size,
    "seed": seed,
})


{'model_name': 'cross-encoder/stsb-distilroberta-base', 'dataset': 'glue/stsb', 'split': 'validation', 'device': 'mps', 'pipeline_device': 'mps', 'batch_size': 64, 'seed': 42}


In [2]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()

def word_count(text):
    return len(re.findall(r"\b\w+\b", str(text)))

df["len1_words"] = df["sentence1"].map(word_count)
df["len2_words"] = df["sentence2"].map(word_count)
df["avg_words"] = (df["len1_words"] + df["len2_words"]) / 2.0
df["len_gap_words"] = (df["len1_words"] - df["len2_words"]).abs()

filtered_df = df[
    df["len1_words"].between(7, 18)
    & df["len2_words"].between(7, 18)
    & df["len_gap_words"] <= 6
].copy()

subset_size = min(300, len(filtered_df))
subset_df = filtered_df.sort_values(
    by=["avg_words", "len_gap_words", "sentence1", "sentence2"],
    ascending=[True, True, True, True],
).head(subset_size).reset_index(drop=True)

print({
    "original_num_examples": len(df),
    "filtered_num_examples": len(filtered_df),
    "subset_num_examples": len(subset_df),
    "columns": subset_df.columns.tolist(),
})
print(subset_df[["sentence1", "sentence2", "label", "len1_words", "len2_words", "avg_words", "len_gap_words"]].head(10))


{'original_num_examples': 1500, 'filtered_num_examples': 1500, 'subset_num_examples': 300, 'columns': ['sentence1', 'sentence2', 'label', 'len1_words', 'len2_words', 'avg_words', 'len_gap_words']}
                 sentence1               sentence2  label  len1_words  \
0   Three men are dancing.      Women are dancing.  1.300           4   
1       Women are running.  Two women are running.  3.824           3   
2         A man is crying.     A woman is dancing.  0.600           4   
3        A man is dancing.       A man is singing.  1.250           4   
4        A man is dancing.      A man is speaking.  1.200           4   
5        A man is running.       A man is singing.  1.250           4   
6       A man is speaking.       A man is dancing.  1.200           4   
7       A man is speaking.      A man is spitting.  0.636           4   
8       A man jumping rope       A man is talking.  0.400           4   
9  A man practicing boxing  A man practices boxing  5.000           4   


In [3]:
length_stats = {
    "len1_words_mean": float(subset_df["len1_words"].mean()),
    "len1_words_median": float(subset_df["len1_words"].median()),
    "len1_words_min": int(subset_df["len1_words"].min()),
    "len1_words_max": int(subset_df["len1_words"].max()),
    "len2_words_mean": float(subset_df["len2_words"].mean()),
    "len2_words_median": float(subset_df["len2_words"].median()),
    "len2_words_min": int(subset_df["len2_words"].min()),
    "len2_words_max": int(subset_df["len2_words"].max()),
    "avg_words_mean": float(subset_df["avg_words"].mean()),
    "avg_words_median": float(subset_df["avg_words"].median()),
    "len_gap_words_mean": float(subset_df["len_gap_words"].mean()),
    "len_gap_words_median": float(subset_df["len_gap_words"].median()),
}
print(length_stats)


{'len1_words_mean': 5.793333333333333, 'len1_words_median': 6.0, 'len1_words_min': 3, 'len1_words_max': 9, 'len2_words_mean': 5.78, 'len2_words_median': 6.0, 'len2_words_min': 3, 'len2_words_max': 8, 'avg_words_mean': 5.786666666666667, 'avg_words_median': 6.0, 'len_gap_words_mean': 0.88, 'len_gap_words_median': 1.0}


In [4]:
pipe = pipeline(
    task="text-classification",
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device,
    truncation=True,
)
print(model_name)
print(pipe.model.config)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cross-encoder/stsb-distilroberta-base
RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "LABEL_0": 0
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.3.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}



In [5]:
sentences1 = subset_df["sentence1"].tolist()
sentences2 = subset_df["sentence2"].tolist()
labels = subset_df["label"].to_numpy(dtype=np.float32)

pair_inputs = [{"text": s1, "text_pair": s2} for s1, s2 in zip(sentences1, sentences2)]

raw_outputs = pipe(
    pair_inputs,
    batch_size=batch_size,
    top_k=None,
)

def output_to_score(output):
    if isinstance(output, list):
        if len(output) == 1 and isinstance(output[0], dict) and output[0].get("label", "").upper() == "LABEL_0":
            score = float(output[0]["score"])
        else:
            label_to_prob = {item["label"].upper(): float(item["score"]) for item in output}
            if "LABEL_1" in label_to_prob and "LABEL_0" in label_to_prob:
                score = label_to_prob["LABEL_1"]
            else:
                score = float(max(output, key=lambda x: x["score"])["score"])
    elif isinstance(output, dict):
        score = float(output["score"])
    else:
        score = float(output)
    return float(np.clip(score, 0.0, 1.0) * 5.0)

predicted_score_0_5 = np.array([output_to_score(x) for x in raw_outputs], dtype=np.float32)
signed_error = predicted_score_0_5 - labels
absolute_error = np.abs(signed_error)


In [6]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic
mae = float(np.mean(absolute_error))

results_df = subset_df.copy()
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["signed_error"] = signed_error
results_df["absolute_error"] = absolute_error

underestimation_df = results_df.sort_values(["signed_error", "absolute_error"], ascending=[True, False]).reset_index(drop=True)
overestimation_df = results_df.sort_values(["signed_error", "absolute_error"], ascending=[False, False]).reset_index(drop=True)

score_diagnostics = {
    "label_min": float(labels.min()),
    "label_max": float(labels.max()),
    "label_mean": float(labels.mean()),
    "prediction_min": float(predicted_score_0_5.min()),
    "prediction_max": float(predicted_score_0_5.max()),
    "prediction_mean": float(predicted_score_0_5.mean()),
    "num_predictions_below_0": int(np.sum(predicted_score_0_5 < 0.0)),
    "num_predictions_above_5": int(np.sum(predicted_score_0_5 > 5.0)),
    "num_predictions_at_0": int(np.sum(np.isclose(predicted_score_0_5, 0.0))),
    "num_predictions_at_5": int(np.sum(np.isclose(predicted_score_0_5, 5.0))),
}

print(results_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "signed_error", "absolute_error"]].head(10))
print(score_diagnostics)
print(underestimation_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "signed_error", "absolute_error", "len1_words", "len2_words"]].head(10))
print(overestimation_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "signed_error", "absolute_error", "len1_words", "len2_words"]].head(10))


                 sentence1               sentence2  label  \
0   Three men are dancing.      Women are dancing.  1.300   
1       Women are running.  Two women are running.  3.824   
2         A man is crying.     A woman is dancing.  0.600   
3        A man is dancing.       A man is singing.  1.250   
4        A man is dancing.      A man is speaking.  1.200   
5        A man is running.       A man is singing.  1.250   
6       A man is speaking.       A man is dancing.  1.200   
7       A man is speaking.      A man is spitting.  0.636   
8       A man jumping rope       A man is talking.  0.400   
9  A man practicing boxing  A man practices boxing  5.000   

   predicted_score_0_5  signed_error  absolute_error  
0             2.118078      0.818078        0.818078  
1             4.572669      0.748669        0.748669  
2             0.026063     -0.573937        0.573937  
3             0.576492     -0.673508        0.673508  
4             0.757092     -0.442909        0.442909 

In [7]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"original_num_examples: {len(df)}")
print(f"filtered_num_examples: {len(filtered_df)}")
print(f"subset_num_examples: {len(subset_df)}")
print(f"subset_rule: sentence1_words_and_sentence2_words_in_[7,18]_and_length_gap<=6_then_sorted_deterministically_head_{len(subset_df)}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"mean_absolute_error: {mae:.6f}")
print(f"prediction_min: {predicted_score_0_5.min():.6f}")
print(f"prediction_max: {predicted_score_0_5.max():.6f}")
print(f"prediction_mean: {predicted_score_0_5.mean():.6f}")
print(f"label_min: {labels.min():.6f}")
print(f"label_max: {labels.max():.6f}")
print(f"label_mean: {labels.mean():.6f}")
print(f"avg_len1_words: {subset_df['len1_words'].mean():.2f}")
print(f"avg_len2_words: {subset_df['len2_words'].mean():.2f}")
print(f"avg_length_gap_words: {subset_df['len_gap_words'].mean():.2f}")
print(f"max_underestimation: {results_df['signed_error'].min():.6f}")
print(f"max_overestimation: {results_df['signed_error'].max():.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")

top_under_examples = underestimation_df[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "signed_error", "absolute_error", "len1_words", "len2_words"
]].head(5)
top_over_examples = overestimation_df[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "signed_error", "absolute_error", "len1_words", "len2_words"
]].head(5)

print({"top_underestimations": top_under_examples.to_dict(orient="records")})
print({"top_overestimations": top_over_examples.to_dict(orient="records")})


device_used: mps
model_name: cross-encoder/stsb-distilroberta-base
dataset_split: glue/stsb/validation
original_num_examples: 1500
filtered_num_examples: 1500
subset_num_examples: 300
subset_rule: sentence1_words_and_sentence2_words_in_[7,18]_and_length_gap<=6_then_sorted_deterministically_head_300
pearson_correlation: 0.922047
spearman_correlation: 0.925215
mean_absolute_error: 0.439993
prediction_min: 0.025722
prediction_max: 4.957826
prediction_mean: 2.396759
label_min: 0.000000
label_max: 5.000000
label_mean: 2.390357
avg_len1_words: 5.79
avg_len2_words: 5.78
avg_length_gap_words: 0.88
max_underestimation: -3.150483
max_overestimation: 2.547709
runtime_seconds: 2.80
{'top_underestimations': [{'sentence1': 'Israel expands subsidies to settlements', 'sentence2': 'Israel widens settlement subsidies', 'label': 5.0, 'predicted_score_0_5': 1.8495171070098877, 'signed_error': -3.1504828929901123, 'absolute_error': 3.1504828929901123, 'len1_words': 5, 'len2_words': 4}, {'sentence1': 'UK al